# Structural Analysis with Identified VARs

This tutorial walks through the complete structural VAR pipeline: fitting a model, applying Cholesky identification, and computing impulse responses, forecast error variance decompositions, and historical decompositions.

In [ ]:
import numpy as np
import pandas as pd

from litterman import VAR, VARData
from litterman.identification import Cholesky
from litterman.samplers import NUTSSampler

## Simulate a VAR(1) process

In [ ]:
rng = np.random.default_rng(42)
T = 200
y = np.zeros((T, 3))
for t in range(1, T):
    y[t, 0] = 0.6 * y[t - 1, 0] + rng.standard_normal() * 0.1
    y[t, 1] = 0.3 * y[t - 1, 0] + 0.5 * y[t - 1, 1] + rng.standard_normal() * 0.1
    y[t, 2] = 0.2 * y[t - 1, 1] + 0.4 * y[t - 1, 2] + rng.standard_normal() * 0.1

index = pd.date_range("2000-01-01", periods=T, freq="QS")
data = VARData(endog=y, endog_names=["gdp", "inflation", "rate"], index=index)

## Fit the model

In [ ]:
sampler = NUTSSampler(draws=500, tune=500, chains=2, cores=1, random_seed=42)
fitted = VAR(lags=2, prior="minnesota").fit(data, sampler=sampler)
fitted

## Apply Cholesky identification

The ordering determines the causal structure. Variables listed first are "most exogenous" — they are not contemporaneously affected by variables listed later.

In [ ]:
identified = fitted.set_identification_strategy(
    Cholesky(ordering=["gdp", "inflation", "rate"])
)
identified

## Impulse response functions

How does a one-standard-deviation structural shock propagate through the system?

In [ ]:
irf = identified.impulse_response(horizon=20)
irf.median()

## Forecast error variance decomposition

What fraction of each variable's forecast error variance is explained by each structural shock?

In [ ]:
fevd = identified.fevd(horizon=20)
fevd.median()

## Historical decomposition

What was the contribution of each structural shock to each variable at each point in time?

In [ ]:
hd = identified.historical_decomposition()
hd.median()